## Resample from 30 m to 100 m

### Author: Celia Baumhoer (2025)

In [4]:
import os
import numpy as np
import rasterio
from rasterio.enums import Resampling
from rasterio.warp import calculate_default_transform, reproject, transform as rio_transform
from rasterio.windows import Window
from rasterio.transform import Affine
from glob import glob
from datetime import datetime
import geopandas as gpd
from shapely.geometry import Point

In [9]:
# define input and output dir on AI4SNOW project directory
# fsc_raw: train data; 21 global & 21 alpine (CH excluded) Landsat fsc scenes
# fsc_raw_rockies: test data in Canada; 8 scenes, June to Sep, no others were available
# fsc_raw_switzerland: test data in Switzerland 9 scenes, one per month between Sep and Apr
in_dir = "/dss/dsstbyfs02/pn49ci/pn49ci-dss-0013/evaluation/Landsat/fsc_raw_switzerland"
resample_dir = "/dss/dsstbyfs02/pn49ci/pn49ci-dss-0013/evaluation/Landsat/fsc_test_switzerland_100m"
patch_dir = f"{resample_dir}_patches")
# define target resolution for resampling
target_resolution = 100  # in meters
# define amount of random points to sample patches from each scene
random_points = 20000

# create output folder
os.makedirs(resample_dir, exist_ok=True)
os.makedirs(patch_dir, exist_ok=True)

## Resample to 100 m resolution

In [10]:
for tif_path in glob(os.path.join(in_dir, "*.tif")):
    with rasterio.open(tif_path) as src:
        data = src.read(1).astype(float)

        # Reclassify
        data_reclassified = np.full(data.shape, np.nan)
        # snow free is class value 10
        data_reclassified[data == 10] = 0.0
        # snow is class value 11
        data_reclassified[data == 11] = 1.0

        # Determine the transform and shape of the output (resampled) raster
        transform, width, height = calculate_default_transform(
            src.crs, src.crs, src.width, src.height, *src.bounds, resolution=target_resolution
        )

        profile = src.profile.copy()
        profile.update({
            'dtype': 'float32',
            'count': 1,
            'compress': 'deflate',
            'nodata': np.nan,
            'transform': transform,
            'width': width,
            'height': height
        })

        dst_data = np.empty((height, width), dtype=np.float32)

        reproject(
            source=data_reclassified,
            destination=dst_data,
            src_transform=src.transform,
            src_crs=src.crs,
            dst_transform=transform,
            dst_crs=src.crs,
            # with the average function FSC might be a bit higher in areas with lots of NAN values
            # e.g. averaging 9 pixels into one 100x100m pixel and 6 are NAN and 3 are snow the entire
            # patch will have a FSC of 1. Mabybe we need to change this.
            resampling=Resampling.average, 
            src_nodata=np.nan,
            dst_nodata=np.nan
        )

        out_path = os.path.join(resample_dir, os.path.basename(tif_path))
        with rasterio.open(out_path, "w", **profile) as dst:
            dst.write(dst_data, 1)